In [1]:
from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Iceberg_Projeto") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/app/data/iceberg_warehouse") \
    .getOrCreate()

print("Spark com Iceberg iniciado")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d617de23-42db-4fba-974d-f6191d3bd934;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
:: resolution report :: resolve 132ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-subm

Spark com Iceberg iniciado


In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS local.db_teste")
print("Database criada")

Database criada


In [3]:
spark.sql("SHOW DATABASES IN local").show()

+---------+
|namespace|
+---------+
| db_teste|
+---------+



In [4]:
data = [
    ("Engenharia", 1),
    ("Dados", 2),
    ("Big Data", 3)
]

df = spark.createDataFrame(data, ["palavra", "valor"])
df.show()

+----------+-----+
|   palavra|valor|
+----------+-----+
|Engenharia|    1|
|     Dados|    2|
|  Big Data|    3|
+----------+-----+



In [5]:
df.writeTo("local.db_teste.tabela_v1").createOrReplace()

print("Tabela Iceberg criada")
spark.table("local.db_teste.tabela_v1").show()

Tabela Iceberg criada
+----------+-----+
|   palavra|valor|
+----------+-----+
|Engenharia|    1|
|     Dados|    2|
|  Big Data|    3|
+----------+-----+



In [6]:
spark.sql("""
INSERT INTO local.db_teste.tabela_v1
VALUES ('Spark', 4)
""")

print("INSERT feito")
spark.table("local.db_teste.tabela_v1").show()

INSERT feito
+----------+-----+
|   palavra|valor|
+----------+-----+
|Engenharia|    1|
|     Spark|    4|
|     Dados|    2|
|  Big Data|    3|
+----------+-----+



In [7]:
spark.sql("""
UPDATE local.db_teste.tabela_v1
SET valor = 99
WHERE palavra = 'Dados'
""")

print("UPDATE feito")
spark.table("local.db_teste.tabela_v1").show()

26/04/20 17:48:51 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


UPDATE feito
+----------+-----+
|   palavra|valor|
+----------+-----+
|     Spark|    4|
|Engenharia|    1|
|  Big Data|    3|
|     Dados|   99|
+----------+-----+



In [8]:
spark.sql("""
DELETE FROM local.db_teste.tabela_v1
WHERE palavra = 'Engenharia'
""")

print("DELETE feito")
spark.table("local.db_teste.tabela_v1").show()

DELETE feito
+--------+-----+
| palavra|valor|
+--------+-----+
|   Spark|    4|
|   Dados|   99|
|Big Data|    3|
+--------+-----+



In [9]:
spark.sql("""
SELECT * FROM local.db_teste.tabela_v1.snapshots
""").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-04-20 17:38:...|3150570446398736522|               NULL|   append|/app/data/iceberg...|{spark.app.id -> ...|
|2026-04-20 17:38:...|3387851444569454316|3150570446398736522|   append|/app/data/iceberg...|{spark.app.id -> ...|
|2026-04-20 17:38:...|5886267370420308067|3387851444569454316|overwrite|/app/data/iceberg...|{spark.app.id -> ...|
|2026-04-20 17:38:...|3271634634173608881|5886267370420308067|   delete|/app/data/iceberg...|{spark.app.id -> ...|
|2026-04-20 17:48:...|2914506756037058025|               NULL|   append|/app/data/iceberg...|{spark.app.id -> ...|
|2026-04-20 17:48:...|4753540458451595613|2914506756037058025|   append|/app/dat

In [12]:
spark.sql("DESCRIBE TABLE local.db_teste.tabela_v1").show()

+--------+---------+-------+
|col_name|data_type|comment|
+--------+---------+-------+
| palavra|   string|   NULL|
|   valor|   bigint|   NULL|
+--------+---------+-------+

